# 05 · Run the whole pipeline

Orchestrates `01` → `02` → `03` → `04` in one go with `notebookutils.notebook.run`, passing the same lakehouse and
schema parameters to every step. Use it once the profiling review and the logical-model overrides have settled and you
want a repeatable rebuild — for a first pass, run the notebooks one at a time and read their summaries.

For production scheduling, wire the same four notebooks into a **Fabric Data Factory pipeline** (four Notebook activities
in sequence, each with these parameters) and schedule the pipeline; this notebook is the interactive equivalent.

In [ ]:
# PARAMETERS — override from a pipeline or notebookutils.notebook.run()
LAKEHOUSE_ROOT = ""            # "" = default lakehouse attached to this notebook (the child notebooks inherit it)
SOURCE_SCHEMAS = ""            # profiling scope, see 01
INCLUDE_TABLES = ""
EXCLUDE_TABLES = "profiling*,dim_*,fact_*,gold*"
PROFILING_SCHEMA = "profiling"
GOLD_SCHEMA = "gold"
MODEL_NAME = "Lakehouse Model"
TABLE_ROLE_OVERRIDES = ""      # see 02
NATURAL_KEY_OVERRIDES = ""
RELATIONSHIP_SCAN_MODE = "name"
RUN_PROFILING = True           # False = reuse the latest profiling run
RUN_MODEL_DESIGN = True        # False = reuse the existing Files/model/model_spec.json (after manual edits)
RUN_GOLD_BUILD = True
RUN_SEMANTIC_MODEL = True
TIMEOUT_SECONDS = 3600

In [ ]:
import time

steps = []
if RUN_PROFILING:
    steps.append(("01_lakehouse_data_profiling", {"LAKEHOUSE_ROOT": LAKEHOUSE_ROOT, "SOURCE_SCHEMAS": SOURCE_SCHEMAS, "INCLUDE_TABLES": INCLUDE_TABLES,
                                                   "EXCLUDE_TABLES": EXCLUDE_TABLES, "OUTPUT_SCHEMA": PROFILING_SCHEMA, "RELATIONSHIP_SCAN_MODE": RELATIONSHIP_SCAN_MODE}))
if RUN_MODEL_DESIGN:
    steps.append(("02_logical_model_design", {"LAKEHOUSE_ROOT": LAKEHOUSE_ROOT, "PROFILING_SCHEMA": PROFILING_SCHEMA, "MODEL_NAME": MODEL_NAME,
                                               "GOLD_SCHEMA": GOLD_SCHEMA, "TABLE_ROLE_OVERRIDES": TABLE_ROLE_OVERRIDES, "NATURAL_KEY_OVERRIDES": NATURAL_KEY_OVERRIDES}))
if RUN_GOLD_BUILD:
    steps.append(("03_build_dim_fact_model", {"LAKEHOUSE_ROOT": LAKEHOUSE_ROOT, "GOLD_SCHEMA": GOLD_SCHEMA}))
if RUN_SEMANTIC_MODEL:
    steps.append(("04_semantic_model_deploy", {"LAKEHOUSE_ROOT": LAKEHOUSE_ROOT, "GOLD_SCHEMA": GOLD_SCHEMA, "SEMANTIC_MODEL_NAME": MODEL_NAME}))

results = []
for nb, params in steps:
    t0 = time.time()
    print(f"▶ {nb} {params}")
    exit_value = notebookutils.notebook.run(nb, TIMEOUT_SECONDS, params)
    results.append((nb, round(time.time() - t0, 1), exit_value))
    print(f"✔ {nb} finished in {results[-1][1]}s")

print("\nPipeline summary")
for nb, secs, ev in results:
    print(f"  {nb:32s} {secs:>8.1f}s  {ev or ''}")